# Lekcija 9: Geometrijsko preslikavanje slike, inverzno preslikavanje i bilinearna interpolacija

U Lekciji 8 izrađivali smo matrice transformacije; ova lekcija obrađuje njihovu stvarnu *primjenu* pri ponovnom uzorkovanju slike. Jednostavan pristup &mdash; premještanje svakog izvornog piksela na njegov transformirani položaj &mdash; pokazuje se neispravnim. Vidjet ćemo zašto, riješiti problem **inverznim preslikavanjem**, a zatim obraditi činjenicu da inverzno preslikavanje daje necjelobrojne koordinate piksela za koje je potrebna *interpolacija*.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
def make_test_image(size=60):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    cv2.rectangle(img, (5, 5), (size - 5, size - 5), (60, 90, 160), -1)   # muted blue
    tri = np.array([[30, 12], [10, 32], [50, 32]], dtype=np.int32)
    cv2.fillPoly(img, [tri], (225, 195, 60))  # muted yellow
    return img

img = make_test_image()
plt.imshow(img)
plt.title('Test image')
plt.axis('off')
plt.show()

## Problem s izravnim preslikavanjem

Očit način geometrijskog preslikavanja slike jest za svaki izvorni piksel $(x,y)$ izračunati odredište $(x',y')=M(x,y)$ i ondje kopirati njegovu vrijednost. To se naziva **izravno preslikavanje** (engl. *forward mapping*).

Ono ima temeljni nedostatak: iako izvorni pikseli čine *potpunu* mrežu, njihovi transformirani položaji uglavnom je ne čine. Kada transformacija povećava sliku ili je rotira, čime se mreža lokalno rasteže duž dijagonale, odredišni položaji se razmiču i ostavljaju praznine &mdash; nijedan izvorni piksel ne pada točno na neke odredišne piksele.


In [ ]:
h, w = img.shape[:2]
M = cv2.getRotationMatrix2D((w / 2, h / 2), angle=25, scale=1.6)

forward = np.zeros_like(img)
ys, xs = np.mgrid[0:h, 0:w]
src_pts = np.stack([xs.ravel(), ys.ravel(), np.ones(xs.size)])
dst_pts = M @ src_pts

dxi = np.round(dst_pts[0]).astype(int)
dyi = np.round(dst_pts[1]).astype(int)
valid = (dxi >= 0) & (dxi < w) & (dyi >= 0) & (dyi < h)
forward[dyi[valid], dxi[valid]] = img[ys.ravel()[valid], xs.ravel()[valid]]

filled = np.count_nonzero(forward.sum(axis=2))
print(f'{filled} / {h*w} destination pixels received a value ({100*filled/(h*w):.0f}%)')

plt.imshow(forward)
plt.title('Forward mapping: visible holes (black speckles)')
plt.axis('off')
plt.show()

## Rješenje: inverzno preslikavanje

Umjesto pitanja „kamo odlazi svaki izvorni piksel?”, za svaki *odredišni* piksel postavimo obrnuto pitanje: „odakle je ovaj piksel došao u izvornoj slici?” To znači primjenu **inverzne** transformacije $M^{-1}$ na svaku odredišnu koordinatu. Budući da sada prolazimo kroz potpunu odredišnu mrežu, svaki izlazni piksel dobiva vrijednost &mdash; sam postupak osigurava da nema praznina.

Poteškoća je u tome što $M^{-1}(x',y')$ gotovo nikada ne daje točno cjelobrojnu koordinatu u izvornoj slici. Vrijednost moramo *interpolirati* iz okolnih izvornih piksela.


In [ ]:
M_inv = cv2.invertAffineTransform(M)

dst_grid = np.stack([xs.ravel(), ys.ravel(), np.ones(xs.size)])
src_coords = M_inv @ dst_grid
src_x = src_coords[0].reshape(h, w)
src_y = src_coords[1].reshape(h, w)

print('example fractional source coordinate for destination pixel (10, 15):')
print(f'  ({src_x[15, 10]:.3f}, {src_y[15, 10]:.3f})')

### Interpolacija najbližim susjedom

Najjednostavnija mogućnost jest zaokružiti koordinate na najbliži izvorni piksel s cjelobrojnim koordinatama. Time nema praznina, ali rezultat je blokovit &mdash; mnogi odredišni pikseli zaokružuju se na *isti* izvorni piksel, pa slika izgleda pikselizirano na mjestima na kojima je transformacija povećava.


In [ ]:
def nearest_sample(image, xf, yf):
    h, w = image.shape[:2]
    xi = np.round(xf).astype(int)
    yi = np.round(yf).astype(int)
    valid = (xi >= 0) & (xi < w) & (yi >= 0) & (yi < h)
    out = np.zeros(xf.shape + (image.shape[2],), dtype=np.uint8)
    out[valid] = image[yi[valid], xi[valid]]
    return out

nearest = nearest_sample(img, src_x, src_y)

plt.imshow(nearest)
plt.title('Inverse mapping + nearest neighbor: no holes, but blocky')
plt.axis('off')
plt.show()

### Bilinearna interpolacija

Umjesto odabira jednog najbližeg piksela, **bilinearna interpolacija** miješa 4 susjedna piksela, ponderirajući ih prema blizini promatranoj točki. Neka je $(x_0,y_0)$ piksel s cjelobrojnim koordinatama neposredno dolje i lijevo od točke $(x,y)$ te neka je $d_x = x-x_0$, $d_y=y-y_0$, pri čemu su $0 \le d_x, dy < 1$ necjelobrojni dijelovi koordinata. Interpolirana vrijednost tada je:

$$I(x,y) \approx (1-d_x)(1-d_y)\,I(x_0,y_0) + d_x(1-d_y)\,I(x_1,y_0) + (1-d_x)\,d_y\,I(x_0,y_1) + d_x\,d_y\,I(x_1,y_1)$$

gdje je $x_1=x_0+1$, $y_1=y_0+1$. To su zapravo dvije uzastopne jednodimenzionalne linearne interpolacije, prvo duž $x$, a zatim duž $y$ &mdash; otuda naziv *bilinearna*.


In [ ]:
def bilinear_sample(image, xf, yf):
    h, w = image.shape[:2]
    x0 = np.floor(xf).astype(int)
    y0 = np.floor(yf).astype(int)
    x1, y1 = x0 + 1, y0 + 1
    dx = (xf - x0)[..., None]
    dy = (yf - y0)[..., None]

    valid = (x0 >= 0) & (x1 < w) & (y0 >= 0) & (y1 < h)
    x0c, x1c = np.clip(x0, 0, w - 1), np.clip(x1, 0, w - 1)
    y0c, y1c = np.clip(y0, 0, h - 1), np.clip(y1, 0, h - 1)

    Ia = image[y0c, x0c].astype(np.float64)
    Ib = image[y0c, x1c].astype(np.float64)
    Ic = image[y1c, x0c].astype(np.float64)
    Id = image[y1c, x1c].astype(np.float64)

    out = (1 - dx) * (1 - dy) * Ia + dx * (1 - dy) * Ib + (1 - dx) * dy * Ic + dx * dy * Id
    out[~valid] = 0
    return np.clip(out, 0, 255).astype(np.uint8)

bilinear = bilinear_sample(img, src_x, src_y)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(nearest)
axes[0].set_title('Nearest neighbor')
axes[0].axis('off')
axes[1].imshow(bilinear)
axes[1].set_title('Bilinear')
axes[1].axis('off')
plt.tight_layout()
plt.show()

Bilinearni rezultat ima glatke rubove oko trokuta i pravokutnika umjesto nazubljenih stepenica &mdash; uobičajen kompromis je mekša, malo zamućenija slika u zamjenu za uklanjanje artefakata aliasinga.


### Provjera usporedbom s bibliotekom OpenCV

Funkcija `cv2.warpAffine` interno radi upravo to &mdash; inverzno preslikavanje i interpolaciju. Uspoređujemo vlastitu implementaciju s funkcijom `cv2.warpAffine(..., flags=cv2.INTER_LINEAR)`, primijenjenom s matricom *izravnog* preslikavanja `M`. OpenCV prema zadanim postavkama interno izračunava njezin inverz.


In [ ]:
cv_result = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR)

diff = np.abs(bilinear.astype(int) - cv_result.astype(int))
print(f'max pixel difference  = {diff.max()}   (out of 255)')
print(f'mean pixel difference = {diff.mean():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [bilinear, cv_result, diff.astype(np.uint8) * 20],
                          ['Our bilinear', 'cv2.warpAffine', 'Difference (x20)']):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Rezultati se gotovo potpuno podudaraju &mdash; preostala mala razlika, najviše jedna razina sivog tona, nastaje zato što OpenCV radi brzine interno koristi zaokruživanje u aritmetici s fiksnim zarezom umjesto pune aritmetike s pomičnim zarezom.


### Zadaci

1. Zašto se problem praznina pri izravnom preslikavanju ne pojavljuje kada transformacija samo *smanjuje* sliku? Isprobajte faktor skaliranja `0.5` umjesto `1.6` u primjeru izravnog preslikavanja i provjerite koji je udio odredišnih piksela popunjen.
2. Razmotrite kako biste funkciju `bilinear_sample` proširili na bikubičnu interpolaciju: umjesto susjedstva veličine 2 × 2, koju biste veličinu susjedstva trebali i koje biste svojstvo željeli da težine zadovoljavaju u točkama uzorkovanja?
3. Usporedite vrijeme izvođenja funkcije `bilinear_sample` s funkcijom `cv2.warpAffine` na većoj slici, primjerice 800 × 800, koristeći `%timeit`. Koliko je inačica iz biblioteke OpenCV brža i zašto?
